In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def get_rmat(yaw):
    y = np.array([0, 0, -1])
    z = np.array([-np.cos(yaw), -np.sin(yaw), 0])
    return np.array([np.cross(y, z), y, z])


f = 22388.125
f_mm = 107.463
w = 1024
h = 512
cx = (w - 1) / 2
cy = (h - 1) / 2
rmats_gt = np.array(
    [get_rmat(theta) for theta in np.deg2rad([-120, -90, -45, 0, 45, 90, 120])]
)
tvecs_gt = np.array([[0, 0, f_mm]] * len(rmats_gt), dtype=float)
pts3d_gt = np.random.default_rng(0).uniform(-0.5, 0.5, size=(100, 3))

In [ ]:
def plot_camera(rmat, tvec, ax3d, length=None):
    center = -rmat.T @ tvec
    if length is None:
        length = np.linalg.norm(tvec) * 0.2
    for i, c in enumerate("rgb"):
        ax3d.plot(*zip(center, center + rmat[i] * length), c=c)


fig, ax3d = plt.subplots(subplot_kw={"projection": "3d"})
for rmat, tvec in zip(rmats_gt, tvecs_gt):
    plot_camera(rmat, tvec, ax3d)
ax3d.set_aspect("equal")

In [ ]:
import numpy as np
from scipy.linalg import expm


def small_rotation(sigma=1e-3, seed=None):
    rng = np.random.default_rng(seed)
    omega = rng.normal(scale=sigma, size=3)  # small rotation vector
    K = np.array(
        [[0, -omega[2], omega[1]], [omega[2], 0, -omega[0]], [-omega[1], omega[0], 0]]
    )
    return expm(K)


rmats = np.array([small_rotation(0.05, i) @ rmat for i, rmat in enumerate(rmats_gt)])
tvecs = tvecs_gt + np.random.default_rng(0).normal(scale=5, size=tvecs_gt.shape)

fig, ax3d = plt.subplots(subplot_kw={"projection": "3d"})
for rmat, tvec in zip(rmats_gt, tvecs_gt):
    plot_camera(rmat, tvec, ax3d)
for rmat, tvec in zip(rmats, tvecs):
    plot_camera(rmat, tvec, ax3d)
ax3d.set_aspect("equal")

In [ ]:
from deeperfly.multiview_geom import rmats2rvecs, project, rvecs2rmats
from deeperfly.bundle_adjustment import bundle_adjustment

rvecs_gt = rmats2rvecs(rmats_gt)
kmat_gt = np.array([[f, 0, cx], [0, f, cy], [0, 0, 1]])
pmats_gt = kmat_gt @ np.concatenate([rmats_gt, tvecs_gt[..., None]], axis=-1)
pts2d_gt = project(pmats_gt, pts3d_gt)

In [ ]:
rvecs = rmats2rvecs(rmats)
fix_rvecs = np.zeros_like(rvecs, dtype=bool)
fix_rvecs[3] = True
fix_tvecs = np.zeros_like(tvecs, dtype=bool)
fix_tvecs[3] = True
results = bundle_adjustment(
    pts2d=pts2d_gt,
    rvecs=rvecs,
    tvecs=tvecs,
    intrinsics=np.array([[f, cx, cy]] * len(rmats_gt)),
    pts3d=pts3d_gt,
    fix_intrinsics=True,
    fix_rvecs=fix_rvecs,
    fix_tvecs=fix_tvecs,
)

In [ ]:
# Align the optimized system to ground truth using camera 3 as the anchor.
# A rigid world transform X_new = R_a @ X_old + t_a induces
#   R_i' = R_i @ R_a.T,  t_i' = t_i - R_i' @ t_a
# Solving R_3' = R_gt[3], t_3' = t_gt[3] gives the values below.
rmats_opt = rvecs2rmats(results["rvecs"])
tvecs_opt = results["tvecs"]
pts3d_opt = results["pts3d"]

R_a = rmats_gt[3].T @ rmats_opt[3]
t_a = rmats_gt[3].T @ (tvecs_opt[3] - tvecs_gt[3])

rmats_opt = rmats_opt @ R_a.T
tvecs_opt = tvecs_opt - np.einsum("oij,j->oi", rmats_opt, t_a)
pts3d_opt = pts3d_opt @ R_a.T + t_a

results["rvecs"] = rmats2rvecs(rmats_opt)
results["tvecs"] = tvecs_opt
results["pts3d"] = pts3d_opt

In [ ]:
rmats_opt = rvecs2rmats(results["rvecs"])
tvecs_opt = results["tvecs"]

fig, ax3d = plt.subplots(subplot_kw={"projection": "3d"})
for rmat, tvec in zip(rmats_gt, tvecs_gt):
    plot_camera(rmat, tvec, ax3d)
for rmat, tvec in zip(rmats_opt, tvecs_opt):
    plot_camera(rmat, tvec, ax3d)
ax3d.set_aspect("equal")
ax3d.view_init(elev=90, azim=0)